# Session 03: Practice Quiz

**Notebook 2 of 2: Self-check** · about 35 minutes

For each question:

1. Write down your answer **before** opening "Show answer", or before running the code cell.
2. Run the cell where there is one.
3. Compare, and make sure you understand the *reasoning* in the answer, not only the result.

This quiz is formative: it's for you, and it isn't graded.

## Q1 · Predict the output

```python
def tag_ticket(ticket_id, tag, tags_by_id={}):
    tags_by_id.setdefault(ticket_id, []).append(tag)
    return tags_by_id

first = tag_ticket("INC-1", "dns")
second = tag_ticket("INC-2", "vpn")
print(second)
print(first is second)
```

In [ ]:
def tag_ticket(ticket_id, tag, tags_by_id={}):
    tags_by_id.setdefault(ticket_id, []).append(tag)
    return tags_by_id


first = tag_ticket("INC-1", "dns")
second = tag_ticket("INC-2", "vpn")
print(second)
print(first is second)

<details>
<summary>Show answer</summary>

```text
{'INC-1': ['dns'], 'INC-2': ['vpn']}
True
```

The default `{}` was created **once**, when `def` ran. Both calls used and returned that same dict, so `first` and `second` are one object, and the second call's result still contains the first call's data.

Fix: `tags_by_id=None`, then `if tags_by_id is None: tags_by_id = {}` inside the function.

</details>

## Q2 · Predict the output

```python
def search(tickets, team, *, status="open", limit=10):
    ...

search(tickets, "platform", status="closed")   # call A
search(tickets, team="platform", limit=5)      # call B
search(tickets, "platform", "closed")          # call C
```

Which calls succeed, and what happens with the others?

<details>
<summary>Show answer</summary>

- **A** succeeds: two positional arguments, then a keyword argument.
- **B** succeeds: `team` can be passed by name, and `limit` is keyword-only.
- **C** fails with `TypeError: search() takes 2 positional arguments but 3 were given`. Everything after `*` is keyword-only.

The error is the feature: a reader of call C couldn't tell what `"closed"` means, and the function's author can now add or reorder options without breaking callers.

</details>

## Q3 · Predict the output

What does this print, and in what order?

In [ ]:
def load(raw):
    try:
        print("try")
        value = int(raw)
    except ValueError:
        print("except")
        return "invalid"
    else:
        print("else")
        return value
    finally:
        print("finally")


print(load("7"))
print(load("seven"))

<details>
<summary>Show answer</summary>

```text
try
else
finally
7
try
except
finally
invalid
```

`else` runs only when the `try` block raised nothing. `finally` runs on **every** path, after the `return` value has been computed but before the function actually returns. That's why `finally` prints before the caller prints the result.

</details>

## Q4 · Reading a traceback

```text
Traceback (most recent call last):
  File "C:\work\incident-report\incident_report.py", line 141, in <module>
    raise SystemExit(main())
  File "C:\work\incident-report\incident_report.py", line 122, in main
    report = build_report(tickets, rejected, sla_hours=config["sla_hours"])
  File "C:\work\incident-report\incident_report.py", line 98, in build_report
    "breached": sum(is_breached(t, sla_hours) for t in tickets),
  File "C:\work\incident-report\ticket_summary.py", line 41, in is_breached
    return is_open(ticket) and ticket["age_hours"] > sla_hours[ticket["priority"]]
KeyError: 'P5'
```

1. What went wrong, in one sentence?
2. In which function and file was the exception raised?
3. Which is the most useful place to fix it, and why?

<details>
<summary>Show answer</summary>

1. A ticket has priority `"P5"`, and the SLA table has no entry for it.
2. In `is_breached`, line 41 of `ticket_summary.py`: the **last** frame, just above the error line.
3. Not in `is_breached`. That function is correct to use `[]` and fail loudly. The real problem is that a record with an unknown priority got past validation. Fix it where the data enters: the validation step should reject unknown priorities, so `build_report` only ever sees valid tickets.

Reading order: last line for **what**, the bottom frame for **where**, and the frames above for **how execution got there**, which is where you decide where the fix belongs.

</details>

## Q5 · Debugging: `UnboundLocalError`

```python
rejected_count = 0

def reject(record):
    if not isinstance(record, dict):
        rejected_count += 1
        return False
    return True

reject(["not", "a", "dict"])
```

Why does this raise `UnboundLocalError`, and what's a better design than adding `global`?

<details>
<summary>Show answer</summary>

Assigning to `rejected_count` anywhere inside `reject` makes it a **local** name for the whole function. `rejected_count += 1` reads that local before it has a value.

Adding `global rejected_count` would make it run, but it hides state in the module: tests interfere with each other, and running the job twice in one process doubles the count. Better: let the function answer one question (is this record valid?), and let the caller count:

```python
rejected_count = sum(not is_valid(record) for record in records)
```

</details>

## Q6 · Predict the output: a JSON round trip

```python
import json

report = {"top_tags": [("network", 4), ("api", 3)], 2026: "year", "assignee": None}
restored = json.loads(json.dumps(report))
print(restored)
print(restored == report)
```

In [ ]:
import json

report = {"top_tags": [("network", 4), ("api", 3)], 2026: "year", "assignee": None}
restored = json.loads(json.dumps(report))
print(restored)
print(restored == report)

<details>
<summary>Show answer</summary>

```text
{'top_tags': [['network', 4], ['api', 3]], '2026': 'year', 'assignee': None}
False
```

- Tuples became **lists**, because JSON has only arrays.
- The integer key `2026` became the **string** `'2026'`, because JSON object keys are always strings.
- `None` survived as `null` and came back as `None`.

If something reads your JSON output later (a test, another service), design the output shape around JSON's types: lists of small dicts such as `{"tag": "network", "count": 4}` are clearer than tuples anyway.

</details>

## Q7 · Debugging: the job that always succeeds

```python
def load_all(paths):
    tickets = []
    for path in paths:
        try:
            with open(path, encoding="utf-8") as file:
                tickets.extend(json.load(file))
        except Exception:
            pass
    return tickets
```

The nightly job using this has never failed, yet some mornings the report is nearly empty. What's wrong, and how would you rewrite the error handling?

<details>
<summary>Show answer</summary>

`except Exception: pass` swallows **every** failure: a missing file, malformed JSON, a permissions problem, even a bug such as a typo in a variable name (`NameError`). Nobody is told, so a nearly empty report looks like a quiet night.

Better:

- Catch only what you expect and can handle: `FileNotFoundError`, `UnicodeDecodeError` and `json.JSONDecodeError` (or `OSError` and `ValueError`, their parent classes).
- **Record** each failure (file name and reason) and return it alongside the tickets, so the report can show "2 files rejected".
- Let anything else propagate. An unexpected exception is a bug, and a crash with a traceback is far better than silently wrong output.
- If no file could be read at all, that's a failure of the job: exit with a non-zero code.

</details>

## Q8 · Implementation: read the on-call rota

`rota.json` maps each team to the engineer on call, for example `{"platform": "amira", "network": "li"}`.

Write `read_rota(path)`. It returns the dict. On any problem it raises `RotaError` with a message that **starts with the file name**, chaining the original exception where there is one:

- the file doesn't exist;
- the file isn't valid JSON (include the line number);
- the top level isn't a JSON object;
- any engineer is not a string (name the team).

The test cell checks all five cases.

In [ ]:
import json
import tempfile
from pathlib import Path


class RotaError(Exception):
    """The on-call rota file could not be used."""


def read_rota(path: Path) -> dict[str, str]:
    ...  # your implementation


with tempfile.TemporaryDirectory() as workdir:
    folder = Path(workdir)
    (folder / "good.json").write_text('{"platform": "amira", "network": "li"}', encoding="utf-8")
    (folder / "broken.json").write_text('{"platform": "amira",\n "network": }', encoding="utf-8")
    (folder / "list.json").write_text('["amira", "li"]', encoding="utf-8")
    (folder / "null.json").write_text('{"platform": "amira", "network": null}', encoding="utf-8")

    print(read_rota(folder / "good.json"))
    for name in ["missing.json", "broken.json", "list.json", "null.json"]:
        try:
            read_rota(folder / name)
            print(f"{name}: no exception raised (expected RotaError)")
        except RotaError as error:
            cause = type(error.__cause__).__name__ if error.__cause__ else None
            print(f"RotaError: {error}  (cause: {cause})")

<details>
<summary>Show answer</summary>

```python
def read_rota(path: Path) -> dict[str, str]:
    try:
        with open(path, encoding="utf-8") as file:
            rota = json.load(file)
    except FileNotFoundError as error:
        raise RotaError(f"{path.name}: file not found") from error
    except json.JSONDecodeError as error:
        raise RotaError(f"{path.name}: invalid JSON at line {error.lineno}: {error.msg}") from error
    if not isinstance(rota, dict):
        raise RotaError(f"{path.name}: expected an object, got {type(rota).__name__}")
    for team, engineer in rota.items():
        if not isinstance(engineer, str):
            raise RotaError(f"{path.name}: engineer for {team!r} must be a string")
    return rota
```

Expected output:

```text
{'platform': 'amira', 'network': 'li'}
RotaError: missing.json: file not found  (cause: FileNotFoundError)
RotaError: broken.json: invalid JSON at line 2: Expecting value  (cause: JSONDecodeError)
RotaError: list.json: expected an object, got list  (cause: None)
RotaError: null.json: engineer for 'network' must be a string  (cause: None)
```

Points to notice:

- The `try` block contains only the open-and-parse step. The shape and value checks come after it, because they can't raise those exceptions.
- The last two errors have no cause: nothing failed at a lower level, the data just isn't what the format promises. So there's no `from`.
- This raises on the **first** bad value. When you validate many records, as in the workbook, you'll collect problems instead, so one run reports everything.

</details>

## Q9 · Engineering judgment: handle it here, or let it propagate?

For each situation, should the function **catch** the exception, or **let it propagate** to its caller?

1. `parse_age(raw: str) -> int` gets `"twelve"` from a record.
2. `load_config(path)` can't find the configuration file.
3. The loop over 40 export files hits one with malformed JSON.
4. `team_summary()` raises `KeyError` because of a typo in a dict key in your own code.
5. The CLI's `main()` gets an exception that means "configuration invalid".

<details>
<summary>Show answer</summary>

1. **Propagate** (as a `ValueError`, possibly with a better message). `parse_age` doesn't know whether a bad age should reject the record, use a default or stop the job. Its caller does.
2. **Propagate**, translated into your own `ConfigError` (chained with `from`), so callers can handle "configuration problem" without knowing about file systems.
3. **Catch**, in the loop. This is the code that owns the batch and knows the policy: reject this file, carry on, report it.
4. **Propagate.** It's a bug. Catching it would hide the bug; the traceback is exactly what you need to fix it.
5. **Catch** in `main()`, print a clear one-line message to stderr, and exit with a non-zero code. The top level of a command-line tool is where expected failures become user-facing messages instead of tracebacks.

The principle: handle an exception in the place that **knows what to do about it**. Everywhere else, let it travel.

</details>

## Q10 · Comparison: cleanup and checked exceptions

1. What are the Java and C# equivalents of `with open(path, encoding="utf-8") as file:`? What does each guarantee?
2. Java forces you to declare or catch checked exceptions such as `IOException`. Python doesn't. What does that mean for how you document and review Python functions?

<details>
<summary>Show answer</summary>

1. Java's try-with-resources (`try (var reader = Files.newBufferedReader(path)) { ... }`) and C#'s `using` statement or declaration. All three guarantee the resource is closed when the block exits, whether normally, by `return`, or by an exception. In Python, any object that implements the context-manager protocol (`__enter__`/`__exit__`) works with `with`.
2. Nothing in the language tells you what a Python function can raise. So:
   - document the exceptions a caller is expected to handle in the docstring (for example, "Raises ExportError if the file is missing or malformed");
   - prefer a small number of domain exception types (`ExportError`, `ConfigError`) over leaking low-level ones;
   - in code review, ask "what happens when this fails?" for every I/O call, because no compiler will.

</details>

## How did you do?

- **Q1, Q2, Q5** cover function design and scope: Parts 1–3 of `01-concepts.ipynb`.
- **Q3, Q4, Q7, Q9** cover exceptions and tracebacks: Part 5.
- **Q6, Q8** cover JSON and files: Parts 6 and 7. The workbook uses the same pattern for export files.
- **Q10** compares Python's approach with Java and C#.

Next: the workbook assignment in [`../workbook/assignment.md`](../workbook/assignment.md).